# Imports

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch

from src.data_loader import orchestrate_data_download
from src.data_preprocessor import orchestrate_data_preprocess
from src.data_collator import create_data_collator
from src.training_arguments import create_training_arguments
from src.model_loader import load_model
from src.model_trainer import create_trainer

# Constants

In [3]:
data_path = "FinanceMTEB/financial_phrasebank"
save_path="financial_phrasebank"

split_ratios={'train': 0.7, 'test': 0.2, 'val': 0.1}

checkpoint_distilbert = "distilbert-base-uncased"
checkpoint_finbert = "ProsusAI/finbert"

# Data download and save

In [4]:
_, num_classes, save_path = orchestrate_data_download(data_path=data_path, split_ratios=split_ratios, save_path=save_path)

Saving the dataset (0/1 shards):   0%|          | 0/1264 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1629 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/182 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/453 [00:00<?, ? examples/s]

Dataset dictionary: DatasetDict({
    train: Dataset({
        features: ['text', 'label_text', 'label'],
        num_rows: 1629
    })
    val: Dataset({
        features: ['text', 'label_text', 'label'],
        num_rows: 182
    })
    test: Dataset({
        features: ['text', 'label_text', 'label'],
        num_rows: 453
    })
})


# Data tokenization

In [5]:
tokenizer_distilbert = orchestrate_data_preprocess(data_path=save_path, checkpoint=checkpoint_distilbert, save_path=f"distilbert_{save_path}")

collator_distilbert = create_data_collator(tokenizer=tokenizer_distilbert)

Map:   0%|          | 0/1629 [00:00<?, ? examples/s]

Map:   0%|          | 0/182 [00:00<?, ? examples/s]

Map:   0%|          | 0/453 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1629 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/182 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/453 [00:00<?, ? examples/s]

Tokenized dataset dictionary: DatasetDict({
    train: Dataset({
        features: ['text', 'label_text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1629
    })
    val: Dataset({
        features: ['text', 'label_text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 182
    })
    test: Dataset({
        features: ['text', 'label_text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 453
    })
})


In [6]:
tokenizer_finbert = orchestrate_data_preprocess(data_path=save_path, checkpoint=checkpoint_finbert, save_path=f"finbert_{save_path}")

collator_finbert = create_data_collator(tokenizer=tokenizer_finbert)

Map:   0%|          | 0/1629 [00:00<?, ? examples/s]

Map:   0%|          | 0/182 [00:00<?, ? examples/s]

Map:   0%|          | 0/453 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1629 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/182 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/453 [00:00<?, ? examples/s]

Tokenized dataset dictionary: DatasetDict({
    train: Dataset({
        features: ['text', 'label_text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1629
    })
    val: Dataset({
        features: ['text', 'label_text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 182
    })
    test: Dataset({
        features: ['text', 'label_text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 453
    })
})


# Training arguments

In [23]:
training_arguments_distilbert = create_training_arguments(output_path=checkpoint_distilbert)

In [24]:
training_arguments_finbert = create_training_arguments(output_path=checkpoint_finbert)

# Model loading

In [25]:
model_distilbert = load_model(checkpoint=checkpoint_distilbert, num_labels=num_classes, device="cuda" if torch.cuda.is_available() else "cpu")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [26]:
model_finbert = load_model(checkpoint=checkpoint_finbert, num_labels=num_classes, device="cuda" if torch.cuda.is_available() else "cpu")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Model training

In [27]:
trainer_distilbert = create_trainer(
    model=model_distilbert, 
    training_arguments=training_arguments_distilbert, 
    data_collator=collator_distilbert, 
    tokenized_datasets_path=f"tokenized_distilbert_{save_path}",
    tokenizer=tokenizer_distilbert
    )

trainer_distilbert.train()

Step,Training Loss,Validation Loss,Accuracy,F1
50,3.929323,0.757776,0.719780,0.460842
100,2.144658,0.405348,0.868132,0.820848
150,0.948242,0.200616,0.934066,0.924079
200,0.556228,0.098644,0.967033,0.960179
250,0.376668,0.127319,0.956044,0.936970
300,0.320384,0.098957,0.967033,0.964889
350,0.193894,0.096123,0.967033,0.964889
400,0.081759,0.065467,0.983516,0.979378
450,0.094578,0.068520,0.983516,0.979378
500,0.064789,0.083326,0.978022,0.974494


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=500, training_loss=0.8720345029830933, metrics={'train_runtime': 539.8929, 'train_samples_per_second': 15.086, 'train_steps_per_second': 0.945, 'total_flos': 93295710151182.0, 'train_loss': 0.8720345029830933, 'epoch': 4.901960784313726})

In [28]:
trainer_finbert = create_trainer(
    model=model_finbert, 
    training_arguments=training_arguments_finbert, 
    data_collator=collator_finbert, 
    tokenized_datasets_path=f"tokenized_finbert_{save_path}",
    tokenizer=tokenizer_finbert
    )

trainer_finbert.train()

Step,Training Loss,Validation Loss,Accuracy,F1
50,8.146513,0.653237,0.703297,0.541843
100,1.583652,0.229070,0.912088,0.855341
150,0.619704,0.079388,0.967033,0.954509
200,0.360650,0.084249,0.978022,0.969161
250,0.158602,0.069278,0.978022,0.972912
300,0.182022,0.060970,0.978022,0.973013
350,0.127339,0.056749,0.983516,0.978043
400,0.055928,0.056518,0.989011,0.983135
450,0.068179,0.054878,0.989011,0.983135
500,0.029544,0.054586,0.989011,0.983135


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=500, training_loss=1.1455381934642792, metrics={'train_runtime': 959.0565, 'train_samples_per_second': 8.493, 'train_steps_per_second': 0.532, 'total_flos': 185305332835854.0, 'train_loss': 1.1455381934642792, 'epoch': 4.901960784313726})

# Other